# PyTorch Foundations 03 - Give the weights a home

Part 02 showed that autograd can follow raw tensors. It won't remember
which loose tensors you intended to update, move, or save. Forget one and
the calculation can still run while part of your model stays behind.

**Guiding question:** What does `nn.Module` organize without changing the math?

Return from word positions to two sensor columns: **engine load** and
**ambient temperature**, with one made-up **fuel-use target**. Parts 04–05
keep those roles while adding batches and validation.

| Complaint | What we will check |
|---|---|
| Loose tensors are easy to forget | Train them first, then register the same calculation in a module |
| A tidy class can still ask the wrong question | Match prediction and target shapes before scoring |
| "Same math" sounds vague | Compare parameters, predictions, and all 180 recorded losses |
| A trained object disappears with its kernel | Save weights plus reconstruction settings for Part 04 |

![Loose fuel-model weights and bias become registered parts of FuelLinearModel without changing its two-input, one-output prediction.](../images/03-parameters-need-a-home.png)

The setup makes 96 rows. Stack load and temperature as **columns**, not
192 separate examples. Before noise, a load of 0.5 and temperature 0 gives
`1.8×0.5 - 0.65×0 + 0.4 = 1.3`; raising temperature to 0.4 gives 1.04.
These are pencil examples of the synthetic rule, not real fuel measurements.

We will prove raw/module parity and write
`artifacts/part03_linear_state.pt` plus `artifacts/part03_config.json`.
Mini-batching, held-out selection, and recovery come later.
[Handwritten theory](03-models-and-training-loops-theory.md).


In [ ]:
import sys
print(f"Kernel interpreter: {sys.executable}")
from pathlib import Path
LESSON_DIR = next(
    folder for folder in (
        Path(globals().get("__vsc_ipynb_file__", "")).resolve().parent,
        Path.cwd(),
        Path.cwd().joinpath('learning', '02-pytorch-foundations', '03-models-and-training-loops'),
    )
    if (folder / "03-models-and-training-loops.ipynb").is_file()
)

# -- Imports, deterministic data, and device ----------------------------------
import json
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(23)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_examples = 96
engine_load = torch.linspace(0.1, 1.0, n_examples)
ambient_temp = torch.linspace(-0.8, 0.9, n_examples).roll(17)
# Two (96 examples,) vectors -> (96, 2): align each row's engine load and
# temperature as feature columns so one prediction uses both measurements.
X = torch.stack([engine_load, ambient_temp], dim=1).to(device)
noise = 0.03 * torch.randn(n_examples, 1, device=device)
y = (1.8 * X[:, :1] - 0.65 * X[:, 1:2] + 0.4 + noise)
print(f"X={tuple(X.shape)} (example, feature); y={tuple(y.shape)} (example, target)")


## 1 - Loose tensors can already make the guess

Give load one adjustable influence, temperature another, and add a baseline:

```text
one input row [load, temperature]
-> two weighted contributions
-> add them and the bias
-> one fuel guess

X [96,2] @ W [2,1] + b [1] -> prediction [96,1]
```

All 96 rows reuse **three parameters**. Starting them at zero makes every
initial guess zero and gives us a controlled comparison with the module
later. `requires_grad=True` lets backward write their sensitivities.

The function receives inputs, weights, and bias explicitly and returns
predictions. It doesn't receive targets or update anything.
**Predict:** Does doubling the row count double the parameter count?
No: the rows change the calculation's activations, not its three knobs.


In [ ]:
# -- Create raw trainable tensors and a visible forward function ---------------
raw_W = torch.zeros(2, 1, device=device, requires_grad=True)
raw_b = torch.zeros(1, device=device, requires_grad=True)

def raw_forward(inputs, W, b):
    # (N rows, 2 features) @ (2, 1 target) -> (N, 1); reuse bias (1,)
    # across rows to add the same baseline fuel use to each prediction.
    return inputs @ W + b

raw_initial = raw_forward(X, raw_W, raw_b)
print(f"Raw forward: {tuple(X.shape)} @ {tuple(raw_W.shape)} + {tuple(raw_b.shape)} -> {tuple(raw_initial.shape)}")


### First turn the knobs ourselves

The next block is the whole training routine without a model class:

1. `raw_forward` uses the current weight and bias to make 96 guesses.
2. MSE compares each guess with its own target and returns one score.
3. Backward fills `raw_W.grad` and `raw_b.grad`.
4. Inside `no_grad`, subtract `learning_rate × gradient` from each parameter.
5. Clear those gradients before the next fresh calculation.

The `no_grad` block prevents the **update itself** becoming another prediction
graph. It does not mean the weights have stopped being trainable.

There are 180 updates. Each stored loss was calculated **before** its
update, so the last printed score is not a fresh score after update 180.
**Predict:** Can these three loose parameters reduce the loss without
`nn.Module`? Read the printed endpoints. If they can, the next complaint is
organization, not a missing learning algorithm.


In [ ]:
# -- Train raw tensors with explicit gradient clearing and updates -------------
loss_function = nn.MSELoss()
raw_history = []
learning_rate = 0.12
for _ in range(180):
    prediction = raw_forward(X, raw_W, raw_b)
    loss = loss_function(prediction, y)
    loss.backward()
    with torch.no_grad():
        raw_W -= learning_rate * raw_W.grad
        raw_b -= learning_rate * raw_b.grad
    raw_W.grad.zero_()
    raw_b.grad.zero_()
    raw_history.append(loss.item())
print(f"Raw loss: {raw_history[0]:.6f} -> {raw_history[-1]:.6f}")
# ravel (2 features, 1 target) -> (2,) only for readable parameter output.
print(f"Raw parameters: W={raw_W.detach().cpu().ravel().tolist()}, b={raw_b.item():.3f}")


### The math worked. Who remembers all the pieces?

With loose tensors, you maintain the update list, device moves, and save
list yourself. `nn.Module` is a labelled equipment case for those pieces.
Putting an `nn.Linear` inside it registers that layer's weight and bias,
so `parameters()`, `to(device)`, and `state_dict()` can find them.

`__init__` creates the layer once; `forward` uses it for one prediction.
An ordinary tensor attribute is not automatically a registered parameter.
The optimizer still needs its own explicit job.

The next class returns `[N,1]` predictions from `[N,2]` inputs. We overwrite
its random initialization with zeros, matching the raw start.
**Predict:** Which two names will `named_parameters()` discover?

> **Visual to add — Three knobs, two storage layouts**
> Put the engine-load weight, temperature weight, and baseline bias in
> a loose tray on the left and a labelled `FuelLinearModel.linear` case on
> the right. Draw raw `W` as two rows by one column and `linear.weight`
> as one row by two columns, with load before temperature in both.
> Send the same `[load, temperature]` card through both to one fuel answer.
> Suggested caption: "The case changes ownership and layout, not the guess."


In [ ]:
# -- Encapsulate the same affine function in nn.Module ------------------------
class FuelLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, inputs):
        # (N sensor rows, 2 features) -> (N, 1 fuel target), matching raw_forward.
        return self.linear(inputs)

torch.manual_seed(23)
model = FuelLinearModel().to(device)
with torch.no_grad():
    model.linear.weight.zero_()
    model.linear.bias.zero_()
print(f"Registered parameters: {list(dict(model.named_parameters()))}")


### Read `Linear(2,1)` from the storage side

The raw calculation stores weights `[2,1]`. PyTorch stores them `[1,2]`:
one output row containing a coefficient for each input feature. Internally
it uses the transpose for the row-vector calculation. The bias is `[1]`.

The class owns three trainable numbers; the training loop owns the targets,
loss, and update policy. Calling `model(X)` uses the former without secretly
performing the latter.

### A missing target column can change the question

For a pencil example, let predictions be `[[1],[2]]` and correct targets
`[[1],[2]]`. Their matching MSE is zero. If targets become `[1,2]`,
broadcasting compares each prediction with **both** answers:

```text
misses [[ 0, -1],     four comparisons instead of two
        [ 1,  0]]    MSE = 0.5 instead of 0
```

With 96 rows, that mistake can create a `[96,96]` comparison. The next
cell asserts prediction/target shape equality **before** trusting a
perfectly calculable scalar loss.


In [ ]:
# -- Verify the forward and loss contracts before optimization ----------------
module_prediction = model(X)
module_loss = loss_function(module_prediction, y)
assert module_prediction.shape == y.shape
print(f"Forward: X{tuple(X.shape)} -> prediction{tuple(module_prediction.shape)}")
print(f"Loss accepts prediction{tuple(module_prediction.shape)} and target{tuple(y.shape)} -> scalar{tuple(module_loss.shape)}")


## 2 - A container isn't an update policy

We now give `SGD` the registered parameters and the same learning rate
`0.12` used by hand. This plain SGD has no momentum or adaptive memory;
its step is the same gradient-scaled subtraction.

![Clear gradients, predict and measure, backpropagate, then update; skipping clearing accumulates feedback.](../optional/framework-translation/images/3.png)

The next cell isolates one update: clear, predict, measure, backward, step.
An independent weight copy lets us check that backward only writes feedback.
The gradient norm measures feedback size; the update norm measures movement.
For plain SGD, the latter follows the learning-rate scale. Adam needn't
have that same relation.

**Predict:** At which line should the live weight stop matching the copy?
Don't credit the class for a change made by the optimizer.


In [ ]:
# -- Isolate one module update and measure every state transition -------------
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
optimizer.zero_grad()
before = model.linear.weight.detach().clone()
prediction = model(X)
loss = loss_function(prediction, y)
first_module_loss = loss.item()
loss.backward()
assert torch.equal(model.linear.weight.detach(), before)
# Weight gradient (1, 2) -> scalar norm: inspect feedback magnitude, not an update.
gradient_norm = model.linear.weight.grad.norm().item()
optimizer.step()
# Weight delta (1, 2) -> scalar norm: inspect how far SGD actually moved.
update_norm = (model.linear.weight.detach() - before).norm().item()
print(f"Gradient norm: {gradient_norm:.6f}")
print(f"Parameter update norm: {update_norm:.6f}")


### Count the inspected step

The module already took one step. Give it **179 more**, not 180, so both
routes finish after 180 updates from zeros.

`module_history` starts with the inspected pre-update loss and then records
the remaining pre-update losses. Same rows, targets, MSE, rate, and budget;
only parameter ownership and the implementation of subtraction differ.

The next block returns no special trainer object; its useful products are
the changed model parameters and the list of observed losses. A low final
loss alone would be a weak parity test, so we'll compare the whole path.


In [ ]:
# -- Train the module with the same data, loss, and learning rate --------------
module_history = [first_module_loss]
for _ in range(179):
    optimizer.zero_grad()
    prediction = model(X)
    loss = loss_function(prediction, y)
    loss.backward()
    optimizer.step()
    module_history.append(loss.item())
print(f"Module final loss: {module_history[-1]:.6f}")
print("The loop keeps model, loss, gradient, and update responsibilities separate.")


## 3 - Compare the contents, not just the curves

Transpose the module's `[1,2]` stored weight to match raw `W [2,1]`.
Then compare three kinds of evidence:

- the largest difference among weights and bias;
- the largest difference among predictions on the same inputs;
- the largest difference anywhere in the two pre-update histories.

Taking a maximum prevents a good average from concealing one bad coordinate.
The cell asserts gaps below `1e-5`. If it fails, inspect orientation,
initialization, update counts, and clearing order before blaming the
container. We are proving parity for this calculation and plain SGD,
not for arbitrary framework defaults.


In [ ]:
# -- Compare learned parameters and predictions -------------------------------
# Stored (1 target, 2 features) -> (2 features, 1 target): align the layouts
# before comparing weights, rather than accidentally broadcasting the difference.
module_W = model.linear.weight.detach().T
module_b = model.linear.bias.detach()
with torch.inference_mode():
    raw_prediction = raw_forward(X, raw_W, raw_b)
    module_prediction = model(X)
# Parameter/prediction differences -> scalar maxima: test the worst coordinate.
max_parameter_gap = max(
    (module_W - raw_W.detach()).abs().max().item(),
    (module_b - raw_b.detach()).abs().max().item(),
)
max_prediction_gap = (module_prediction - raw_prediction).abs().max().item()
print(f"Maximum parameter gap: {max_parameter_gap:.2e}")
print(f"Maximum prediction gap: {max_prediction_gap:.2e}")
assert max_parameter_gap < 1e-5 and max_prediction_gap < 1e-5
history_gap = max(abs(raw - module) for raw, module in zip(raw_history, module_history))
print(f"Maximum pre-update history gap: {history_gap:.2e}")
assert len(raw_history) == len(module_history) == 180 and history_gap < 1e-5


Plot the two **recorded** histories together. The dashed line helps you see
the module trace when it sits on the raw trace. The numerical history check
above is stronger evidence than a convincing overlap by eye.

The horizontal index counts loss checks before updates, beginning at zero.
The log vertical scale reveals small later errors; it doesn't convert MSE
into a percentage. The plot changes no model state.

An overlapping path settles the math question. It doesn't solve the next
problem: another notebook cannot borrow this kernel's trained object.


In [ ]:
# -- Plot both recorded optimization histories --------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(raw_history, label="raw tensors")
ax.plot(module_history, linestyle="--", label="nn.Module")
ax.set(xlabel="Loss check before update", ylabel="MSE", title="Same mathematics, different state ownership")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
print("Overlapping histories are evidence of implementation parity.")


## 4 - Save contents and leave rebuilding instructions

Notebooks don't share memory. A state dictionary supplies the model's
named tensors, not its Python class or the meaning of each input column.

The next cell writes two files:

| File | What the next kernel gets |
|---|---|
| `part03_linear_state.pt` | `linear.weight` and `linear.bias` |
| `part03_config.json` | Seed, row count, input/output widths, learning rate |

Part 04 supplies the same model class and **load-then-temperature** feature
order before loading. The JSON does not encode that feature order for us.
Swapping the columns would keep every shape valid while changing the answer.

> **Visual to add — Pack the knobs, keep the assembly card**
> Show the three fuel-model parameters packed into `part03_linear_state.pt`
> and an adjacent card labelled `part03_config.json`: seed 23, 96 rows,
> 2 inputs, 1 output, rate 0.12. Across a "new kernel" gap, show an empty
> `FuelLinearModel` being rebuilt before the weights go in. Keep the
> sensor columns labelled load, temperature. Suggested caption:
> "Saved weights are contents, not a complete program."

Writing files doesn't train again. These files also don't save an optimizer
trajectory; Part 04 checks prediction reload separately from continuation.


In [ ]:
# -- Persist model state and explicit experiment configuration -----------------
artifact_dir = LESSON_DIR / "artifacts"
artifact_dir.mkdir(exist_ok=True)
state_path = artifact_dir / "part03_linear_state.pt"
config_path = artifact_dir / "part03_config.json"
torch.save(model.state_dict(), state_path)
config = {
    "seed": 23,
    "n_examples": n_examples,
    "input_features": 2,
    "output_features": 1,
    "learning_rate": learning_rate,
}
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Saved state: {state_path}")
print(f"Saved configuration: {config_path}")


### Your turn - change the rate, not the starting line

Leave the saved model alone. Each trial below creates a fresh model with
zero weights and bias, then makes ten full-batch updates on the same fuel
rows. Only `YOUR_LEARNING_RATE` changes relative to the reference `0.12`.

**Predict:** Will the supplied `0.03` cover as much ground in that budget?
Compare the printed losses rather than just asking whether both fell.
Their last recorded scores are pre-step checks, like the main histories.
If you choose the reference rate again, the result dictionary has one key,
not two different experiments.

This is a local rate comparison on training data, not held-out selection.
Next we keep the model's math but stop delivering the entire table at once.


In [ ]:
# -- CHANGE THIS: compare one optimizer setting for ten updates ---------------
YOUR_LEARNING_RATE = 0.03
assert YOUR_LEARNING_RATE > 0
trial_results = {}
for rate in (learning_rate, YOUR_LEARNING_RATE):
    trial = FuelLinearModel().to(device)
    with torch.no_grad():
        trial.linear.weight.zero_()
        trial.linear.bias.zero_()
    trial_optimizer = torch.optim.SGD(trial.parameters(), lr=rate)
    trial_losses = []
    for _ in range(10):
        trial_optimizer.zero_grad()
        trial_loss = loss_function(trial(X), y)
        trial_loss.backward()
        trial_optimizer.step()
        trial_losses.append(trial_loss.item())
    trial_results[rate] = trial_losses[-1]
    print(f"Ten-update pre-step loss at lr={rate}: {trial_losses[0]:.4f} -> {trial_losses[-1]:.4f}")
print("Lower final recorded error:", min(trial_results, key=trial_results.get))


## The short version

- Raw tensors already learn. A module makes their ownership discoverable.
- Three parameters serve all 96 fuel rows. Activations depend on the
  current rows; registered weights and bias survive across calls.
- `Linear(2,1)` stores `[1,2]` weights. Compare with raw `[2,1]` weights
  only after transposing.
- Shape equality matters before loss reduction. A missing target column
  can quietly turn paired errors into all-against-all errors.
- The inspected step counts. Both implementations get 180 updates, and
  their parameters, predictions, and histories are checked numerically.
- Weights cross a kernel boundary only with enough rebuilding information.
  Their file doesn't contain the feature names or a training process.

| Coverage | Where we stopped |
|---|---|
| Built and measured | Raw training; registered module; forward/loss shapes; isolated SGD step; full-path parity; real histories; saved weights/settings; rate drill |
| Explained | Parameter ownership; activations versus stored state; optimizer memory; storage orientation |
| Later | Dataset/DataLoader and reload in Part 04; validation and selection in Part 05; schedulers and resumable process state need more than these two files |

**Next:** [Data, Checkpoints, and Hyperparameters](../04-data-checkpoints-and-hyperparameters/04-data-checkpoints-and-hyperparameters.ipynb)
reloads the fuel model, delivers smaller groups, and separates "same
predictions" from "same next update".
